# PyTorch第一次作业代码



## 依赖库安装说明

需要安装的依赖库：
- torch: PyTorch核心库，用于张量操作和神经网络
- torchvision: PyTorch视觉库，包含数据集、模型和图像变换
- matplotlib: 用于图像展示和保存
- pandas: 用于CSV数据读取和处理
- pillow: 用于图像读取（PIL库）


In [10]:
# 导入必要的库
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt

# 加上这两行配置，解决中文显示问题
plt.rcParams['font.sans-serif'] = ['SimHei']  # 指定默认字体为黑体 (SimHei)
plt.rcParams['axes.unicode_minus'] = False    # 解决负号 '-' 显示为方块的问题

import pandas as pd
from PIL import Image
import os

# 关闭matplotlib交互模式，避免阻塞
plt.ioff()

print("库导入完成")

库导入完成


## 模块1：Lenna图像卷积操作

1. 读取当前目录下的lenna.jpg图像
2. 用PyTorch的nn.Conv2d实现卷积操作，卷积核选用3x3边缘检测卷积核
3. 做好维度转换：兼容PIL读取的HWC格式，转为PyTorch卷积要求的BCHW格式
4. 用matplotlib展示并保存卷积后的图像，控制台输出卷积前后的张量shape信息

In [11]:
def module1_lenna_convolution():
    """
    模块1：Lenna图像卷积操作
    读取lenna.jpg，进行卷积操作，展示结果。
    """
    print("=== 模块1：Lenna图像卷积操作 ===")

    # 1. 读取当前目录下的lenna.jpg图像
    image_path = "lenna.jpg"  # 相对路径
    if not os.path.exists(image_path):
        print(f"错误：找不到文件 {image_path}")
        return

    # 使用PIL读取图像，格式为HWC (Height, Width, Channels)
    image = Image.open(image_path).convert('RGB')  # 确保是RGB格式
    print(f"原始图像尺寸：{image.size}")  # (width, height)

    # 转换为PyTorch张量，格式仍为HWC
    transform = transforms.ToTensor()  # 转换为[0,1]范围的张量，格式CHW
    image_tensor = transform(image)  # 现在是CHW格式
    print(f"转换为张量后的shape：{image_tensor.shape}")  # [C, H, W]

    # 转换为BCHW格式，添加batch维度
    image_tensor = image_tensor.unsqueeze(0)  # [1, C, H, W]
    print(f"添加batch维度后的shape：{image_tensor.shape}")

    # 2. 用PyTorch的nn.Conv2d实现卷积操作
    # 选用3x3边缘检测卷积核（Sobel算子近似）
    # 卷积核作用：检测图像中的边缘，特别是水平和垂直边缘
    conv_kernel = torch.tensor([[[[-1, -1, -1],
                                  [0, 0, 0],
                                  [1, 1, 1]],
                                 [[-1, -1, -1],
                                  [0, 0, 0],
                                  [1, 1, 1]],
                                 [[-1, -1, -1],
                                  [0, 0, 0],
                                  [1, 1, 1]]]], dtype=torch.float32)  # 形状：[out_channels, in_channels, kH, kW]

    # 创建Conv2d层，输入通道3，输出通道1，卷积核大小3x3
    conv_layer = nn.Conv2d(in_channels=3, out_channels=1, kernel_size=3, bias=False)
    # 设置卷积核权重
    conv_layer.weight.data = conv_kernel

    # 应用卷积
    conv_output = conv_layer(image_tensor)
    print(f"卷积后的shape：{conv_output.shape}")  # [1, 1, H-2, W-2] (由于padding=0)

    # 3. 维度转换：转回可展示的图像格式
    # 移除batch维度，转换为[H, W]或[H, W, C]
    conv_image = conv_output.squeeze(0).squeeze(0).detach().numpy()  # [H, W]

    # 4. 用matplotlib展示并保存卷积后的图像
    plt.figure(figsize=(10, 5))

    # 原始图像
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("原始Lenna图像")
    plt.axis('off')

    # 卷积后图像
    plt.subplot(1, 2, 2)
    plt.imshow(conv_image, cmap='gray')  # 灰度图
    plt.title("卷积后图像 (边缘检测)")
    plt.axis('off')

    plt.tight_layout()
    plt.savefig("lenna_convolution_result.png", dpi=300, bbox_inches='tight')
    plt.close()

    print("卷积前张量shape：", image_tensor.shape)
    print("卷积后张量shape：", conv_output.shape)
    print("图像已保存为 lenna_convolution_result.png")

    return conv_output  # 返回给模块2使用

# 运行模块1
conv_result = module1_lenna_convolution()

=== 模块1：Lenna图像卷积操作 ===
原始图像尺寸：(512, 512)
转换为张量后的shape：torch.Size([3, 512, 512])
添加batch维度后的shape：torch.Size([1, 3, 512, 512])
卷积后的shape：torch.Size([1, 1, 510, 510])
卷积前张量shape： torch.Size([1, 3, 512, 512])
卷积后张量shape： torch.Size([1, 1, 510, 510])
图像已保存为 lenna_convolution_result.png


## 模块2：池化操作

基于模块1卷积输出的特征图，分别实现2种池化：最大值池化、自适应平均池化。
用PyTorch的nn.MaxPool2d和nn.AdaptiveAvgPool2d实现，参数选用入门常用配置。
完成张量维度处理，用matplotlib分别展示、保存两种池化后的图像。

In [12]:
def module2_pooling(conv_output):
    """
    模块2：池化操作
    基于模块1的卷积输出，进行最大值池化和自适应平均池化。
    """
    print("\n=== 模块2：池化操作 ===")

    if conv_output is None:
        print("错误：模块1输出为空")
        return

    print(f"输入特征图shape：{conv_output.shape}")

    # 1. 最大值池化
    # nn.MaxPool2d(kernel_size=2, stride=2)
    # kernel_size=2: 池化窗口大小2x2
    # stride=2: 步长为2，每次移动2个像素
    max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
    max_pooled = max_pool(conv_output)
    print(f"最大值池化后shape：{max_pooled.shape}")

    # 2. 自适应平均池化
    # nn.AdaptiveAvgPool2d(output_size=(10, 10))
    # output_size=(10, 10): 输出固定大小10x10，无论输入尺寸
    adaptive_pool = nn.AdaptiveAvgPool2d(output_size=(10, 10))
    adaptive_pooled = adaptive_pool(conv_output)
    print(f"自适应平均池化后shape：{adaptive_pooled.shape}")

    # 转换为numpy数组用于展示
    max_img = max_pooled.squeeze(0).squeeze(0).detach().numpy()
    adaptive_img = adaptive_pooled.squeeze(0).squeeze(0).detach().numpy()

    # 3. 用matplotlib展示并保存
    plt.figure(figsize=(15, 5))

    # 原始卷积输出
    plt.subplot(1, 3, 1)
    plt.imshow(conv_output.squeeze(0).squeeze(0).detach().numpy(), cmap='gray')
    plt.title("卷积输出")
    plt.axis('off')

    # 最大值池化
    plt.subplot(1, 3, 2)
    plt.imshow(max_img, cmap='gray')
    plt.title("最大值池化")
    plt.axis('off')

    # 自适应平均池化
    plt.subplot(1, 3, 3)
    plt.imshow(adaptive_img, cmap='gray')
    plt.title("自适应平均池化")
    plt.axis('off')

    plt.tight_layout()
    plt.savefig("pooling_results.png", dpi=300, bbox_inches='tight')
    plt.close()

    print("池化前shape：", conv_output.shape)
    print("最大值池化后shape：", max_pooled.shape)
    print("自适应平均池化后shape：", adaptive_pooled.shape)
    print("图像已保存为 pooling_results.png")

# 运行模块2
module2_pooling(conv_result)


=== 模块2：池化操作 ===
输入特征图shape：torch.Size([1, 1, 510, 510])
最大值池化后shape：torch.Size([1, 1, 255, 255])
自适应平均池化后shape：torch.Size([1, 1, 10, 10])
池化前shape： torch.Size([1, 1, 510, 510])
最大值池化后shape： torch.Size([1, 1, 255, 255])
自适应平均池化后shape： torch.Size([1, 1, 10, 10])
图像已保存为 pooling_results.png


## 模块3：图像批量加载

本地加载：读取当前目录下的imagedata文件夹，用torchvision的ImageFolder+DataLoader实现本地图像批量加载。
打印每一批数据的shape，展示一个批次内的图像样例。

In [13]:
def module3_batch_loading():
    """
    模块3：图像批量加载
    分别从网络和本地加载图像数据，实现批量加载。
    """
    print("\n=== 模块3：图像批量加载 ===")

    # 1. 网络加载：CIFAR10数据集
    print("网络加载：CIFAR10数据集")
    try:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

        # 下载训练集
        train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                                    download=True, transform=transform)

        # DataLoader
        # batch_size=4: 每批4个样本
        # shuffle=True: 随机打乱数据
        # num_workers=2: 使用2个子进程加载数据
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)

        # 获取一个批次
        data_iter = iter(train_loader)
        images, labels = next(data_iter)
        print(f"网络数据批次shape：{images.shape}")  # [batch_size, C, H, W]

        # 展示一个批次的图像样例
        plt.figure(figsize=(12, 3))
        for i in range(4):
            plt.subplot(1, 4, i+1)
            # 反归一化并转换为HWC格式
            img = images[i] / 2 + 0.5  # 反归一化
            img = img.permute(1, 2, 0)  # CHW -> HWC
            plt.imshow(img)
            plt.title(f"标签: {labels[i].item()}")
            plt.axis('off')
        plt.suptitle("网络加载：CIFAR10批次样例")
        plt.tight_layout()
        plt.savefig("network_batch_sample.png", dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"网络加载失败：{e}")
        print("跳过网络加载部分")

    # 2. 本地加载：imagedata文件夹
    print("\n本地加载：imagedata文件夹")
    local_data_path = "imagedata"  # 相对路径
    if not os.path.exists(local_data_path):
        print(f"错误：找不到文件夹 {local_data_path}")
        return

    # ImageFolder自动从文件夹结构推断类别
    local_dataset = ImageFolder(root=local_data_path, transform=transforms.Compose([
        transforms.Resize((224, 224)),  # 统一尺寸
        transforms.ToTensor()
    ]))

    # DataLoader
    local_loader = DataLoader(local_dataset, batch_size=4, shuffle=True, num_workers=0)  # 本地用0个worker

    # 获取一个批次
    try:
        data_iter_local = iter(local_loader)
        images_local, labels_local = next(data_iter_local)
        print(f"本地数据批次shape：{images_local.shape}")

        # 展示一个批次的图像样例
        plt.figure(figsize=(12, 3))
        for i in range(min(4, len(images_local))):
            plt.subplot(1, 4, i+1)
            img = images_local[i].permute(1, 2, 0)  # CHW -> HWC
            plt.imshow(img)
            plt.title(f"类别: {labels_local[i].item()}")
            plt.axis('off')
        plt.suptitle("本地加载：imagedata批次样例")
        plt.tight_layout()
        plt.savefig("local_batch_sample.png", dpi=300, bbox_inches='tight')
        plt.close()
    except StopIteration:
        print("本地数据批次为空")

# 运行模块3
module3_batch_loading()


=== 模块3：图像批量加载 ===
网络加载：CIFAR10数据集


C:\Users\steven\.conda\envs\myenv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


网络数据批次shape：torch.Size([4, 3, 32, 32])

本地加载：imagedata文件夹
本地数据批次shape：torch.Size([4, 3, 224, 224])


## 模块4：CSV文本数据预处理

读取当前目录下csvtextdata文件夹里的csv文件，完成完整的基础文本预处理流程。
控制台输出预处理前后的数据样例和核心信息。

In [14]:
def module4_csv_preprocessing():
    """
    模块4：CSV文本数据预处理
    读取csvtextdata文件夹中的CSV文件，进行预处理。
    """
    print("\n=== 模块4：CSV文本数据预处理 ===")

    csv_folder = "csvtextdata"  # 相对路径
    if not os.path.exists(csv_folder):
        print(f"错误：找不到文件夹 {csv_folder}")
        return

    # 查找CSV文件
    csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]
    if not csv_files:
        print("错误：文件夹中没有CSV文件")
        return

    # 读取第一个CSV文件作为示例
    csv_path = os.path.join(csv_folder, csv_files[0])
    print(f"读取CSV文件：{csv_path}")

    # 1. 读取CSV
    df = pd.read_csv(csv_path)
    print("预处理前数据基本信息：")
    print(f"数据形状：{df.shape}")
    print(f"列名：{list(df.columns)}")
    print("前5行数据样例：")
    print(df.head())

    # 2. 缺失值处理
    print(f"\n缺失值统计：\n{df.isnull().sum()}")
    # 简单填充缺失值
    df = df.fillna('unknown')  # 简单填充所有缺失值
    print("缺失值处理后：")
    print(f"剩余缺失值：{df.isnull().sum().sum()}")

    # 3. 文本内容清洗（假设有文本列）
    text_columns = [col for col in df.columns if df[col].dtype == 'object']
    if text_columns:
        text_col = text_columns[0]
        print(f"\n清洗文本列：{text_col}")
        print("清洗前样例：")
        print(df[text_col].head())

        # 简单文本清洗：转小写、移除特殊字符
        df[text_col] = df[text_col].str.lower().str.replace(r'[^\\w\\s]', '', regex=True)
        print("清洗后样例：")
        print(df[text_col].head())

    # 4. 转为PyTorch张量格式
    # 将label列转换为张量
    if 'label' in df.columns:
        labels = torch.tensor(df['label'].values, dtype=torch.long)
        print(f"\n转换为PyTorch张量：")
        print(f"标签张量shape：{labels.shape}")
        print(f"标签张量类型：{labels.dtype}")
        print("标签样例（前5个）：")
        print(labels[:5])
    else:
        print("没有label列")
        labels = None

    print("预处理完成")
    return df, labels

# 运行模块4
df_processed, tensor_data = module4_csv_preprocessing()


=== 模块4：CSV文本数据预处理 ===
读取CSV文件：csvtextdata\test.csv
预处理前数据基本信息：
数据形状：(4, 2)
列名：['label', 'text']
前5行数据样例：
   label                                               text
0      0  set skenbart follows failed swedish book edito...
1      0  dreadful film doctor goes fishing winds catchi...
2      1  saw film sneak preview delightful cinematograp...
3      1  bill paxton taken true story us golf open made...

缺失值统计：
label    0
text     0
dtype: int64
缺失值处理后：
剩余缺失值：0

转换为PyTorch张量：
标签张量shape：torch.Size([4])
标签张量类型：torch.int64
标签样例（前5个）：
tensor([0, 0, 1, 1])
预处理完成


## 总结

所有模块运行完成！

生成的图像文件：
- lenna_convolution_result.png (模块1)
- pooling_results.png (模块2)
- network_batch_sample.png (模块3网络加载)
- local_batch_sample.png (模块3本地加载)

可以查看这些图像来了解每个模块的效果。